# Fake satellites: why direct and fit disagree for m12i "all stars"

Same smooth fake halo as `fake_data_velocity_maps.ipynb` (Alex's `generate_fake_data`, N = 1000, true R = (20, −10, 25) km/s), plus **fake satellites**: cold clumps of stars at one sky position, all moving with one bulk velocity. The reflex is applied to satellite stars too (`v_obs = v_intrinsic − R`), exactly as to halo stars, so the true R is the same in every case.

Cases
1. **smooth**: no satellite (control; both methods should agree)
2. **one satellite** (A)
3. **two satellites, momenta cancel** (A and B move in opposite directions; the m12i situation at a 150 kpc inner edge)
4. **two satellites, momenta add** (same as 3 but B moves the same way as A)

For each case: direct = −⟨v⟩ (number-weighted), fit = Alex's likelihood (`dipole.lnlike`, as in `03_fit_m12i.py`), and the predicted gap
fit − direct ≈ μ_r⟨r̂⟩ + μ_θ⟨θ̂⟩ + μ_φ⟨φ̂⟩ (the "leakage", exact for isotropic σ).

In [ ]:
import os, sys, types
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

sys.path.insert(0, os.path.abspath('../code'))
try:
    import dipole
except ImportError:
    # off SCC: stand-ins for packages the fake-data functions never use
    for name in ['gizmo_analysis', 'halo_analysis', 'utilities', 'utilities.coordinate', 'pygaia']:
        sys.modules.setdefault(name, types.ModuleType(name))
    sys.modules['utilities'].coordinate = sys.modules['utilities.coordinate']
    sys.modules['pygaia'].proper_motion_uncertainty = None
    import dipole

## Parameters — edit here

In [ ]:
N_HALO = 1000
SEED = 1
R_TRUE = np.array([20., -10., 25.])    # true reflex (Cartesian, km/s)
MEAN = np.array([0., 0., 0.])
SIGMA = np.array([100., 80., 80.])
PARS = np.r_[R_TRUE, MEAN, np.log10(SIGMA)]

# satellites: n stars, centre position [kpc], bulk velocity in the halo frame [km/s],
# spatial size and internal 1D dispersion. Halo stars sit at r ~ 50-120 kpc (Gaussian ball, sigma 50 kpc).
SAT_A = dict(name='A', n=300, pos=[80., 0., 0.], vel=[0., 150., 0.], size=3., disp=10.)
SAT_B = dict(name='B', n=300, pos=[-40., 60., -45.], vel=[0., -150., 0.], size=3., disp=10.)
SAT_B_SAME = dict(SAT_B, vel=[0., 150., 0.])

CASES = [('smooth', []),
         ('one satellite (A)', [SAT_A]),
         ('two satellites, momenta cancel', [SAT_A, SAT_B]),
         ('two satellites, momenta add', [SAT_A, SAT_B_SAME])]

CMAP = 'viridis'
POINT_SIZE = 8
FLIP_LON = False

## Helpers

In [ ]:
def sph_basis(xyz):
    """Unit vectors r_hat, theta_hat, phi_hat at each position (rows = stars)."""
    x, y, z = xyz.T
    r = np.linalg.norm(xyz, axis=1)
    th, ph = np.arccos(z / r), np.arctan2(y, x)
    r_hat = xyz / r[:, None]
    th_hat = np.stack([np.cos(th) * np.cos(ph), np.cos(th) * np.sin(ph), -np.sin(th)], axis=1)
    ph_hat = np.stack([-np.sin(ph), np.cos(ph), np.zeros_like(ph)], axis=1)
    return r_hat, th_hat, ph_hat


def sph_to_xyz(v_sph, xyz):
    """Inverse of dipole.vel_xyz_to_sph_optimized."""
    r_hat, th_hat, ph_hat = sph_basis(xyz)
    return v_sph[:, :1] * r_hat + v_sph[:, 1:2] * th_hat + v_sph[:, 2:3] * ph_hat


def direct_v_travel(v_xyz):
    """-(mean Cartesian velocity). Uses no positions at all."""
    return -v_xyz.mean(axis=0)


def fit_v_travel(xyz, v_sph):
    """Alex's likelihood, SLSQP on the per-star mean -lnL (same as scc/03_fit_m12i.py)."""
    n = len(xyz)
    initial = [0, 0, 0, 0, 0, 0, np.log10(50), np.log10(50), np.log10(50)]
    soln = minimize(lambda p: -dipole.lnlike(p, xyz, v_sph) / n, initial, bounds=dipole.get_bounds(),
                    method='SLSQP', options={'maxiter': 500, 'ftol': 1e-10})
    return soln.x


def describe(vec):
    v = np.linalg.norm(vec)
    return f'({vec[0]:6.1f}, {vec[1]:6.1f}, {vec[2]:6.1f})  |v| = {v:5.1f} km/s'


def sky_angles(xyz):
    """Longitude (azimuth phi) and latitude (90 deg - theta) in radians, for a Mollweide plot."""
    x, y, z = xyz.T
    lon = np.arctan2(y, x)
    lat = np.arcsin(z / np.linalg.norm(xyz, axis=1))
    return (-lon if FLIP_LON else lon), lat


def make_satellite(sat, rng):
    """Cold clump: Gaussian in position and velocity; reflex added like the halo stars."""
    xyz = np.asarray(sat['pos']) + rng.normal(0, sat['size'], (sat['n'], 3))
    v_int = np.asarray(sat['vel']) + rng.normal(0, sat['disp'], (sat['n'], 3))
    return xyz, v_int - R_TRUE


def build(sats, n_halo=N_HALO, seed=SEED, scale=1):
    """Smooth fake halo + satellites (scale multiplies satellite star numbers)."""
    xyz, v_sph = dipole.generate_fake_data(PARS, size=(n_halo, 3), seed=seed)
    v_xyz = sph_to_xyz(v_sph, xyz)
    rng = np.random.default_rng(seed + 100)
    label = np.zeros(n_halo, int)
    for k, s in enumerate(sats, start=1):
        x, v = make_satellite(dict(s, n=s['n'] * scale), rng)
        xyz, v_xyz = np.r_[xyz, x], np.r_[v_xyz, v]
        label = np.r_[label, np.full(len(x), k)]
    return xyz, v_xyz, dipole.vel_xyz_to_sph_optimized(v_xyz, xyz), label


def analyse(xyz, v_xyz, v_sph):
    d = direct_v_travel(v_xyz)
    p = fit_v_travel(xyz, v_sph)
    rh, th, ph = sph_basis(xyz)
    leak = p[3] * rh.mean(0) + p[4] * th.mean(0) + p[5] * ph.mean(0)
    return d, p, leak

## 1. Estimates for each case (N_halo = 1000)

In [ ]:
results = {}
for name, sats in CASES:
    xyz, v_xyz, v_sph, label = build(sats)
    d, p, leak = analyse(xyz, v_xyz, v_sph)
    results[name] = (xyz, v_xyz, v_sph, label, d, p, leak)
    f_sat = np.mean(label > 0)
    print(f'--- {name}: {len(xyz)} stars, satellite fraction {f_sat:.0%}')
    print('   true    ', describe(R_TRUE))
    print('   direct  ', describe(d))
    print('   fit     ', describe(p[:3]))
    print(f'   fit mean flow mu = ({p[3]:.0f}, {p[4]:.0f}, {p[5]:.0f}) km/s, fit sigma = {np.round(10**p[6:], 0)}')
    print(f'   fit - direct = {np.round(p[:3] - d, 1)}   predicted leakage = {np.round(leak, 1)}\n')

## 2. Velocity maps of the data

Scatter maps of the observed (v_r, v_θ, v_φ), one row per case. Satellites are the tight single-colour clumps. Markers: **★ true R**, **● direct**, **✕ fit** (sky direction each vector points to).

In [ ]:
def mark(ax, R, **kw):
    lo, la = sky_angles(np.atleast_2d(R))
    ax.plot(lo, la, ls='none', **kw)

MARKS = [('true', lambda r: r[0], dict(marker='*', ms=17, color='k', mec='w')),
         ('direct', lambda r: r[1], dict(marker='o', ms=11, mfc='none', mec='m', mew=2.2)),
         ('fit', lambda r: r[2], dict(marker='X', ms=12, color='r', mec='w'))]
VLIM = 150

fig, axes = plt.subplots(len(CASES), 3, figsize=(15, 4.3 * len(CASES)), subplot_kw={'projection': 'mollweide'})
fig.subplots_adjust(left=0.02, right=0.98, top=0.95, bottom=0.05, wspace=0.08, hspace=0.75)
for axrow, (name, _) in zip(axes, CASES):
    xyz, v_xyz, v_sph, label, d, p, leak = results[name]
    lon, lat = sky_angles(xyz)
    order = np.argsort(label)                       # draw satellites on top
    for k, (ax, lab) in enumerate(zip(axrow, [r'$v_r$', r'$v_\theta$', r'$v_\phi$'])):
        sc = ax.scatter(lon[order], lat[order], c=v_sph[order, k], s=POINT_SIZE, cmap=CMAP, vmin=-VLIM, vmax=VLIM)
        for _, get, kw in MARKS:
            mark(ax, get((R_TRUE, d, p[:3])), **kw)
        ax.set_title(lab, fontsize=13)
        ax.set_xticklabels([]); ax.set_yticklabels([]); ax.grid(alpha=0.3)
        cb = fig.colorbar(sc, ax=ax, orientation='horizontal', fraction=0.05, pad=0.04, label='km/s')
    top, bottom = axrow[1].get_position().y1, cb.ax.get_position().y0
    fig.text(0.5, top + 0.025, name, ha='center', va='bottom', fontsize=14)
    fig.text(0.5, bottom - 0.04,
             f'true |R| = {np.linalg.norm(R_TRUE):.1f}     direct = ({d[0]:.1f}, {d[1]:.1f}, {d[2]:.1f}), |R| = {np.linalg.norm(d):.1f}'
             f'     fit = ({p[0]:.1f}, {p[1]:.1f}, {p[2]:.1f}), |R| = {np.linalg.norm(p[:3]):.1f}  km/s',
             ha='center', va='top', fontsize=12)
handles = [plt.Line2D([], [], ls='none', label=n, **kw) for n, _, kw in MARKS]
fig.legend(handles=handles, loc='upper left', bbox_to_anchor=(0.02, 0.995), ncol=3, frameon=False, fontsize=12)
plt.show()

## 3. Does it go away with more stars?

Same cases with 100× more halo stars **and** 100× more satellite stars (same satellite fractions). Shot noise drops by 10×; a systematic gap stays.

In [ ]:
for name, sats in CASES:
    xyz, v_xyz, v_sph, label = build(sats, n_halo=100 * N_HALO, scale=100)
    d, p, leak = analyse(xyz, v_xyz, v_sph)
    print(f'{name:32s} direct |R| = {np.linalg.norm(d):5.1f}   fit |R| = {np.linalg.norm(p[:3]):5.1f}   '
          f'|fit - direct| = {np.linalg.norm(p[:3] - d):5.1f}   |predicted leakage| = {np.linalg.norm(leak):5.1f}')